In [ ]:
import pandas as pd
import numpy as np
import re

## Charger le CSV

In [ ]:
url= "Pokemon_dataset.csv"
# df = pd.read_csv(url, sep=";", encoding="utf-8")
df = pd.read_csv(url, sep=";", encoding="utf-8", index_col=0)
df

## Transposee du dataframe

In [ ]:
df = df.T
# df.columns = df.iloc[0]
# df = df.iloc[1:].reset_index(drop=True)
df

## Fonction pour voir pokemon par id

In [ ]:
def afficher_pokemon(df, num = []):
    display(df[df['#'].isin(num)])

## Fix les colonnes numeriques

In [ ]:
# ATTENTION : avec cette fonction on va enlever le '-' ce qui va nous donner en quelques sortes
# la valeur absolue des nombres concernés
def regex_garder_numerique(df, columns=[]):
    for name in columns:
        df[name] = df[name].str.replace(r"[^0-9.]", "", regex=True)
    return df

df = regex_garder_numerique(df, ["HP", "Attack", "Defense % of Attack", "Speed", "Generation"])

## Changer les types des colonnes

In [ ]:
colonnes_num = [
    "#", "Total", "HP", "Attack",
    "Defense % of Attack", "Speed", "Generation"
]

for col in colonnes_num:
    df[col] = pd.to_numeric(df[col], errors="raise")
    # erreurs = df[pd.to_numeric(df[col], errors="coerce").isna()][col]
    # if not erreurs.empty:
    #     print(f"\nColonne : {col}")
    #     print(erreurs.unique())

In [ ]:
df.info()

## Trier par # et fix "Generation"

In [ ]:
df.sort_values("#", inplace=True)
df["Generation"] = pd.to_numeric(df["Generation"].ffill().bfill())
df

## Gerer Noms

In [ ]:
df["Name"] = df["Name"].str.title()
df

## Supprimer doublons

In [ ]:
print(df.shape)
df = df.drop_duplicates()
print(df.shape)

In [ ]:
df

## Reset index

In [ ]:
df = df.reset_index(drop=True)
df.columns.name = "index"
df.head()

## Split Sp. Atk / Sp. Def

Cette facon de faire va poser un problème car on ne gère pas les lignes qui ont 3 valeurs dans Sp. Atk / Sp. Def. Cela va nous conduire ensuite a les remplacer par des nan ce qui va apauvrir notre base de données

In [ ]:
# split_cols = df['Sp. Atk / Sp. Def'].astype(str).str.split('/', n=1, expand=True)
# df['Sp. Atk'] = split_cols[0].str.strip()
# df['Sp. Def'] = split_cols[1].str.strip()

# df.drop(columns=['Sp. Atk / Sp. Def'], inplace=True)
# df.head()

### Bonne facon de faire en changeant a la main les deux lignes problématiques

In [ ]:
# # Pour trouver les lignes problématiques
# split = df["Sp. Atk / Sp. Def"].str.split("/", expand=True)
# print(split.shape)

In [ ]:
# # Si le nombre de colonnes est différent de 2, jecherche les valeurs anormales :
# df[df["Sp. Atk / Sp. Def"].str.count("/") != 1]["Sp. Atk / Sp. Def"]
# erreurs = df[df["Sp. Atk / Sp. Def"].str.count("/") != 1]
# print(erreurs)


In [ ]:
# # faut corriger d'abord
# df.loc[789, "Sp. Atk / Sp. Def"] = "44 / 46"
# df.loc[115 , "Sp. Atk / Sp. Def"] = "35 / 110" 
# split = df["Sp. Atk / Sp. Def"].str.split("/", expand=True)

# print(split.shape)
# print(split.columns)
# nb = df["Sp. Atk / Sp. Def"].str.split("/").str.len()

# print(df.loc[nb != 2, "Sp. Atk / Sp. Def"])

In [ ]:
# # nettoyage de données, la meilleure pratique est de les séparer en deux colonnes numériques.
# df[["Sp. Atk", "Sp. Def"]] = df["Sp. Atk / Sp. Def"].str.split("/", expand=True)
# #J' enleve les espaces et convertir en nombres :
# df["Sp. Def"] = pd.to_numeric(df["Sp. Def"].str.strip())
# df["Sp. Atk"] = pd.to_numeric(df["Sp. Atk"].str.strip())

# # Ensuite, je supprime l'ancienne colonne :
# df = df.drop(columns=["Sp. Atk / Sp. Def"])
# df.head()

In [ ]:
# afficher_pokemon(df, [107,713])

### Autre bonne facon de faire avec la regex

In [ ]:
df["Sp. Atk / Sp. Def"] = (
    df["Sp. Atk / Sp. Def"]
    .str.replace("//", "/", regex=False)
    .str.replace(r"\s+", " ", regex=True)
)

df["Sp. Atk / Sp. Def"] = df["Sp. Atk / Sp. Def"].str.replace(
    r"^\s*(\d+)\s*/\s*(\d+)\s*/\s*\d+\s*$",
    r"\1 / \2",
    regex=True
)

df_sp_atk = df["Sp. Atk / Sp. Def"].str.split(" / ", expand=True)[0]
df_sp_def = df["Sp. Atk / Sp. Def"].str.split(" / ", expand=True)[1]

index_sp_atk_def = df.columns.get_loc("Sp. Atk / Sp. Def")

df.insert(index_sp_atk_def + 1, "Sp. Atk", df_sp_atk)
df.insert(index_sp_atk_def + 2, "Sp. Def", df_sp_def)
df.head()

In [ ]:
afficher_pokemon(df, [107,713])

## Fix Defense %

In [ ]:
df.head()

In [ ]:
df["Defense"] = (
    df["Defense % of Attack"] * df["Attack"] / 100
).round()
df.head()

## Valeurs manquantes

In [ ]:
df.isna().sum()

In [ ]:
df["Sp. Atk"] = df["Sp. Atk"].astype(float)
df["Sp. Def"] = df["Sp. Def"].astype(float)
df.info()

### HP manquants

In [ ]:
display(df[(df["HP"] > df["Total"]) | (df["Attack"] > df["Total"]) | (df["Speed"] > df["Total"] ) | (df["Sp. Atk"] > df["Total"] ) | (df["Sp. Def"] > df["Total"] )])
df.loc[df["HP"] > df["Total"], "HP"] = np.nan

display(df[(df["HP"] > df["Total"]) | (df["Attack"] > df["Total"]) | (df["Speed"] > df["Total"]) | (df["Sp. Atk"] > df["Total"] ) | (df["Sp. Def"] > df["Total"] )])


#display(df.loc[df["Id"] == 15])
print(df.isna().sum())

In [ ]:
# Cas deja géré plus haut avec la regex qui enleve les "-"
# display(df.head())
# colonnes = ["Total", "HP", "Defense % of Attack", "Sp. Atk", "Sp. Def", "Speed"]
# display(df[(df[colonnes] < 0).any(axis=1)])
# df["HP"] = df["HP"].abs() # pour mettre la colonne en valeur absolue
# df

In [ ]:
df.fillna({
    "HP": df["Total"] - (df["Attack"] + df["Defense"] + df["Sp. Atk"] + df["Sp. Def"] + df["Speed"] )
}, inplace= True)
df.isna().sum()

### Speed manquants

In [ ]:
df.fillna({
    "Speed": df["Total"] - (df["Attack"] + df["Defense"] + df["Sp. Atk"] + df["Sp. Def"] + df["HP"] )
}, inplace= True)
df.isna().sum()

In [ ]:
df.isna().sum()

### Cas de plusieurs nan par ligne (sauf les Attack Defense qui manquent)

In [ ]:
stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]

def calculer_statistiques_multiple_nan(df, stats, df_ratio):
    masque = df[stats].isna().sum(axis=1) > 1
    print("========== masque ==========")
    print(masque)
    reste = df.loc[masque, "Total"] - df.loc[masque, stats].sum(axis=1)
    print("========== reste ==========")
    print(reste)
        
    ratios = df_ratio.loc[df.loc[masque, "Types"]]
    print("========== ratios ==========")
    print(ratios)

    ratios.index = df.loc[masque].index
    print("========== ratios ==========")
    print(ratios)

    ratios_nan = ratios.where(df.loc[masque, stats].isna())
    print("========== ratios_nan ==========")
    print(ratios_nan)

    nouveaux_ratios = ratios_nan.div(ratios_nan.sum(axis=1), axis=0)
    print("========== nouveaux_ratios ==========")
    print(nouveaux_ratios)
    
    nouvelles_stats = nouveaux_ratios.mul(reste, axis=0)
    print("========== nouvelles_stats ==========")
    print(nouvelles_stats)

    df.loc[masque, stats] = df.loc[masque, stats].fillna(nouvelles_stats.round())

    return df

moyennes_stats = df.groupby(["Types"])[stats].mean()
moyennes_total = df.groupby(["Types"])["Total"].mean()

print(moyennes_stats)
print(moyennes_total)

df_ratio = pd.DataFrame(index=moyennes_stats.index)
df_ratio["HP"] = moyennes_stats["HP"] / moyennes_total
df_ratio["Attack + Defense"] = ((moyennes_stats["Attack"] + moyennes_stats["Defense"]) / moyennes_total)
df_ratio["Sp. Atk"] = moyennes_stats["Sp. Atk"] / moyennes_total
df_ratio["Sp. Def"] = moyennes_stats["Sp. Def"] / moyennes_total
df_ratio["Speed"] = moyennes_stats["Speed"] / moyennes_total

print(df_ratio)

df["Attack + Defense"] = (df["Attack"] + df["Defense"]).where(df["Attack"].notna() & df["Defense"].notna())
new_stats = ["HP", "Attack + Defense", "Sp. Atk", "Sp. Def", "Speed"]

df = calculer_statistiques_multiple_nan(df, new_stats, df_ratio)

In [ ]:
df.isna().sum()

### Cas special : Attack et Defense qui manquent


- x = Attack
- y = Defense
- x + y = Total - sum_stats_no_attack
- def = Defense % of Attack/100
- y = x * def
- x + x*def = Total - sum_stats_no_attack
- 1x + def*x = Total - sum_stats_no_attack
- (1 + def) * x = Total - sum_stats_no_attack
- x = (Total - sum_stats_no_attack) / (1 + def)

In [ ]:
stats_no_attack = ['HP', 'Speed', 'Sp. Atk', 'Sp. Def']
missing_atk = ((df['Total'] - df[stats_no_attack].sum(axis=1)) / (1+df['Defense % of Attack']/100)).round()
df['Attack'] = df['Attack'].fillna(missing_atk)

In [ ]:
df.isna().sum()

### Defense manquantes

In [ ]:
df["Defense"] = (
    df["Defense % of Attack"] * df["Attack"] / 100
).round()
df.isna().sum()

### drop Attack + Defense

In [ ]:
df.drop(columns=["Attack + Defense"], inplace=True)

## fix lignes problematiques restantes

In [ ]:
df.head()

In [ ]:
stats = ['HP', 'Speed', 'Sp. Atk', 'Sp. Def', "Attack", "Defense"]

df[abs(df["Total"] - df[stats].sum(axis=1)) > 0.001] #[stats].sum(axis=1)

In [ ]:
stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
display(df[df[stats].sum(axis=1) != df["Total"]])

df.loc[df[stats].sum(axis=1) != df["Total"], "Attack"] = np.nan
df.loc[df[stats].sum(axis=1) != df["Total"], "Defense"] = np.nan
df.loc[df[stats].sum(axis=1) != df["Total"], "Defense % of Attack"] = np.nan

# display(df[df["#"] == 410])
afficher_pokemon(df, num = [410, 328])

## Calcule la moyenne "Attack" et "Defense" par rapport "Generation"
df_mean = df.groupby("Generation")[["Attack", "Defense"]].agg("mean").round(2)
df_mean["Attack Percent"] = (df_mean["Attack"] / (df_mean["Defense"]+df_mean["Attack"])).round(2)
df_mean["Defense Percent"] = (df_mean["Defense"] / (df_mean["Defense"]+df_mean["Attack"])).round(2)
display(df_mean)

Attack_Defense = df["Total"] - df[["HP", "Sp. Atk", "Sp. Def", "Speed"]].sum(axis=1)
df.fillna({
    "Attack": (df["Generation"].map(df_mean["Attack Percent"]) * Attack_Defense).round(0),
    "Defense": (df["Generation"].map(df_mean["Defense Percent"]) * Attack_Defense).round(0),
}, inplace=True)

# display(df[df["#"] == 410])
afficher_pokemon(df, num = [410, 328])

## ReCalculer le pourcentage de Defense par rapport a l'Attack et Defense de nouvelle valeur: (defe/atta)*100 = defe%
df.fillna({
    "Defense % of Attack": (df["Defense"] / df["Attack"]) * 100
}, inplace=True)

# display(df[df["#"] == 410])
afficher_pokemon(df, num = [410, 328])

In [ ]:
df[abs(df["Total"] - df[stats].sum(axis=1)) > 0.001]

In [ ]:
df.isna().sum()

In [ ]:
df.describe()

In [ ]:
df[df["Defense % of Attack"] == 2300.0]

# <span style="color:#F54927"> GRAPHIQUES sur la correction

In [ ]:
import matplotlib.pyplot as plt

### Diagramme nbre de pokémon par génération

#### données

In [ ]:
obj = df["Generation"].value_counts().sort_index()

generation = obj.index
nbr_pokemon_par_gen = obj.values
display(obj)

generation = np.array(generation)
nbr_pokemon_par_gen = np.array(nbr_pokemon_par_gen)

# display(x, y)

In [ ]:
# df["Legendary"] = df["Legendary"].str.strip().map({"True": True, "False": False})

# la colonne legendaire n'est pas nettoyée = restée une string donc besoin de passer en bool avec un mp sur un dico car astype transformerait toutes les string (soit tout) en True, donc tous les pokemons deveindraient légendaire
# besoin de cleaner les espaces avant de mapper 
# tout ça pour pouvoir sum les bool !!!

# legendaires_par_gen2 = df["Legendary"].groupby(df["Generation"]).sum()
# display(legendaires_par_gen2)

# ---------------- OU utilisation de .xs pour rester en multidimension et pouvoir compter ces putains de strings True dans Legendary
# Cf. cellule suivante

In [ ]:
obj2 = df["Legendary"].groupby(df["Generation"]).value_counts().sort_index()
# display(obj2)

legendaires_par_gen =  obj2.xs("True", level="Legendary")
# non_legendaires_par_gen =  obj2.xs("False", level="Legendary")

legendaires_par_gen = np.array(legendaires_par_gen)

total_pokemon = len(df)
total_legendaire = legendaires_par_gen.sum()
total_non_leg = total_pokemon - total_legendaire


#### histogramme

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(18, 10))

x = generation
y = nbr_pokemon_par_gen

axs[0].plot(x, y, marker="o", linestyle="")
axs[0].set_ylim(0, max(y)+11)

axs[1].bar(x, y)
plt.show()

### Camembert génération / nbre légendaire

In [ ]:
from matplotlib.patches import ConnectionPatch

# pie chart : légendaire vs non-légendaire (global)
overall_ratios = [total_legendaire / total_pokemon, total_non_leg / total_pokemon]
labels = ['Légendaire', 'Non-légendaire']
explode = [0.1, 0]  # on éclate la part "Légendaire"

# bar chart : répartition des légendaires par génération

ratios = legendaires_par_gen / total_legendaire
gen_labels = [f"Gen {g}" for g in generation]

# --- Figure ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 5))
fig.subplots_adjust(wspace=0)

# rotate so that first wedge is split by the x-axis
angle = -180 * overall_ratios[0]
pie = ax1.pie(overall_ratios, autopct='%1.1f %%', colors=['#F5B027', '#F87C63'], startangle=angle,
              labels=labels, explode=explode, #shadow=True
              )

bottom = 1
width = .2
for j, (height, label) in enumerate(#reversed
    ([*zip(ratios, gen_labels)])):
    bottom -= height
    bc = ax2.bar(0, height, width, bottom=bottom, color='#F5276C', label=label,
                 alpha=0.1 + 0.8 * j / len(ratios))
    ax2.bar_label(bc, labels=[f"{height:.0%}"], label_type='center')

ax2.set_title('Pokémon légendaires')
ax2.legend()
ax2.axis('off')
ax2.set_xlim(-2.5 * width, 2.5 * width)

# connecting lines (identique à l'exemple)
theta1, theta2 = pie.wedges[0].theta1, pie.wedges[0].theta2
center, r = pie.wedges[0].center, pie.wedges[0].r
bar_height = sum(ratios)

x = r * np.cos(np.pi / 180 * theta2) + center[0]
y = r * np.sin(np.pi / 180 * theta2) + center[1]
con = ConnectionPatch(xyA=(-width / 2, bar_height), coordsA=ax2.transData,
                      xyB=(x, y), coordsB=ax1.transData)
con.set_color('black')
con.set_linewidth(2)
ax2.add_artist(con)

x = r * np.cos(np.pi / 180 * theta1) + center[0]
y = r * np.sin(np.pi / 180 * theta1) + center[1]
con = ConnectionPatch(xyA=(-width / 2, 0), coordsA=ax2.transData,
                      xyB=(x, y), coordsB=ax1.transData)
con.set_color('black')
con.set_linewidth(2)
ax2.add_artist(con)

plt.show()

# inverser opcaité barre + ombre camembert // delimiter // titre

### Diagramme valeurs stats / nombre de pokémons

In [ ]:
stats_cols = np.array(stats)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 15))
axes = axes.flatten()

for i, col in enumerate(stats_cols):
    axes[i].hist(df[col], color="#BCAFDA", edgecolor="#6576B8", bins="auto")
    axes[i].set_title(col)
    axes[i].set_xlabel("valeur de la stat")
    axes[i].set_ylabel("nombre de pokémon")

plt.tight_layout() # recalcule automatiquement l'espacement entre les sous-graphiques (subplots) et les marges de la figure
# = évite que les titres, labels d'axes ou légendes se chevauchent ou soient coupés = 0 impact sur les données
plt.show()

### Boîtes à stachmou des stats

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
df[stats_cols].boxplot(ax=ax)
# boxplot() méthode pandas =rracourci qui appelle matplotlib.pyplot.boxplot() en extrayant automatiquement les colonnes du DataFrame
# evite de faire :
# data = [df[col].dropna() for col in stats_cols] # liste de tableaux, un par stat
# ax.boxplot(data, label=stats_cols.all())

ax.set_title("Distribution des stats")
ax.set_ylabel("Valeur des stats")
plt.show()

### Violins stats / legendaire ou non

#### Avec Matplotlib

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

positions_legendaire = np.arange(len(stats_cols)) - 0.2
positions_non_legendaire = np.arange(len(stats_cols)) + 0.2

data_legendaires = [df[df["Legendary"] == "True"][col].dropna() for col in stats_cols]
data_non_legendaires = [df[df["Legendary"] == "False"][col].dropna() for col in stats_cols]

parts_legendaire = ax.violinplot(data_legendaires, positions=positions_legendaire, widths=0.35)
parts_non_legendaire = ax.violinplot(data_non_legendaires, positions=positions_non_legendaire, widths=0.35)

for pk in parts_legendaire['bodies']:
    pk.set_facecolor("gold")
    pk.set_alpha(0.6)
for pk in parts_non_legendaire['bodies']:
    pk.set_facecolor("steelblue")
    pk.set_alpha(0.6)

ax.set_xticks(np.arange(len(stats_cols)))
ax.set_xticklabels(stats_cols)
ax.set_ylabel("valeur")
ax.set_title("Stats : Pokémons légendaire VS non-légendaire")

# légende manuelle
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="gold", label="Légendaires"), Patch(color="steelblue", label="Non-légendaires")])

plt.tight_layout()
plt.show()

#### Avec Seaborn

In [ ]:
import seaborn as sns
sns.set_theme() #applique theme par defaut seaborn = fond gris, taille police...
# theme applicable à tous les objets instanciés suivants même les matplotlib natifs (puisque seaborn consruit sur maplotlib et modifie params visuels de mpl)

In [ ]:
# transforamtion en "format long" pour pouvoir utiliser Seaborn correctement avec melt() = dépile plusieurs colonnes en une seule colonne de valeurs + une col qui indique d'où vient chaque valeur
# cf screenshot cours matplotlib section seaborn

df_long = df.melt(
    id_vars="Legendary", # colonnes a garder telles quelles (ici Legendary reste identifiable pour chaque valeur peu importe la stat)
    # = colonnes répétées pour chaque ligne générée
    value_vars=stats_cols, # colonnes à dépiler = transfomer en une lignes (ici hp, def... chaque stat)
    var_name="stats", # nom nouvelle colonne contenant nom d'origine colonne dépliée
    value_name="valeur" # nom nouvelle col qui contient les valeurs
)

# display(df.shape)
# display(df_long.shape)
# display(df.head())
# display(df_long.head())

fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(data=df_long, x="stats", y="valeur", hue="Legendary", split=True, ax=ax, palette=["#FC8DB5", "#F94024"])
# hue = groupement par couleur qui evite de devoir positionner violons manuellement, côte à côte, avec mpl
ax.set_title("Stats : Légendaire vs Non-légendaire")
plt.show()

### Diagrammes types / générations

In [ ]:
type1 = (df["Types"].str.split(",", expand=True))[0]
counts = df.groupby(["Generation", type1]).size().unstack(fill_value=0)
# counts --> lignes = générations, colonnes = types

#### Avec Matplotlib

In [ ]:
generations = counts.index
types = counts.columns

palette = plt.get_cmap("tab20", len(types))
colors = [palette(i) for i in range(len(types))]

fig, ax = plt.subplots(figsize=(14, 7))
bottom = np.zeros(len(generations))

for i, t in enumerate(types):
    values = counts[t].values
    ax.bar(generations, values, bottom=bottom, color=colors[i], label=t)
    bottom += values  # on remonte le "plancher" pour la prochaine couche

ax.set_xlabel("Génération")
ax.set_ylabel("Nombre de Pokémon")
ax.set_title("Nombre de Pokémon par type et génération")
ax.legend(title="Type", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

#### Avec Seaborn

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
counts.plot(kind="bar", stacked=True, ax=ax, colormap="tab20b")
ax.set_xlabel("Génération")
ax.set_ylabel("Nombre de Pokémon")
ax.legend(title="Type", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.show()

### FONCTION RADAR

In [ ]:
from typing import Mapping
from matplotlib.axes import Axes

def radar_chart(
        data: pd.DataFrame,
        entities: list[str], 
        axes: Axes,
        colors: Mapping[str, str | tuple], # tuple pour rgba 
        cols_to_plot: list[str] | np.ndarray,
        title: str,
        alpha_fill: float =0.13
        ) -> None:
    """
    data         : df indexé par une colonne (nom, type, catégorie...) = identifiant 
    entities     : liste des identifiants à tracer (doivent exister dans data.index)
    axes           : axe matplotlib (projection polaire)
    colors       : dict {identifiant: couleur}
    cols_to_plot : liste des colonnes numériques à représenter sur le radar
    title        : titre du graphique
    alpha_fill   : opacité du remplissage
    """

    angles = np.linspace(0, 2 * np.pi, len(cols_to_plot), endpoint=False).tolist()
    # angles positionne chaque stat régulièrement autour du cercle (2π divisé en 6 pour 6 stats).
    # Le += angles[:1] et row[:1] dupliquent le premier point à la fin, pour que la ligne se referme visuellement (sinon trou entre la dernière et la première stat)
    angles += angles[:1]

    
    for entity in entities:
        row = data.loc[entity, cols_to_plot].tolist()
        row += row[:1]
        color = colors.get(entity, "gray") # gris par défaut si le pokémon n'est pas dans le dico
        axes.plot(angles, row, label=entity, linewidth=2, color=color)
        axes.fill(angles, row, alpha=alpha_fill, color=color)

    axes.set_xticks(angles[:-1])
    axes.set_xticklabels(cols_to_plot)
    axes.set_title(title)
    axes.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))    

### Radars évolutions et comparaison de celles-ci

#### Données et dico couleurs

In [ ]:
df_indexed = df.set_index("Name")

In [ ]:
colors_evolution = {
    "Bulbasaur": "#1e5f2c",   
    "Ivysaur": "#5ba85f",    
    "Venusaur": "#9add91",   

    "Charmander": "#e8111c",  
    "Charmeleon": "#e74f25",
    "Charizard": "#f17e58", 

    "Squirtle": "#1f4e8c", 
    "Wartortle": "#5c9bd1",  
    "Blastoise": "#92c9eb", 
}

colors_comparatif = {
    "Venusaur": "#5ba85f",   
    "Charizard": "#e74f25", 
    "Blastoise": "#5c9bd1", 
}

#### En précisant bien l'indice de chaque axe

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11), subplot_kw=dict(polar=True))

# ------------------ INSTANCIATION DES GRAPHS ------------------
# ------------------ 3 évolutions individuelles ------------------
radar_chart(
    data=df_indexed,
    entities=["Bulbasaur", "Ivysaur", "Venusaur"],
    axes=axes[0, 0],
    colors=colors_evolution,
    cols_to_plot=stats_cols,   
    title="Évolution Bulbizarre"
)
radar_chart(
    data=df_indexed,
    entities=["Charmander", "Charmeleon", "Charizard"],
    axes=axes[0, 1],
    colors=colors_evolution,
    cols_to_plot=stats_cols,   
    title="Évolution Bulbizarre"
)
radar_chart(
    data=df_indexed,
    entities=["Squirtle", "Wartortle", "Blastoise"],
    axes=axes[1, 0],
    colors=colors_evolution,
    cols_to_plot=stats_cols,   
    title="Évolution Bulbizarre"
)

# ------------------ comparatif des formes finales ------------------
radar_chart(
    data=df_indexed,
    entities=["Venusaur", "Charizard", "Blastoise"],
    axes=axes[1, 1],
    colors=colors_evolution,
    cols_to_plot=stats_cols,   
    title="Comparatif formes finales", 
    alpha_fill=0.17
)

#### Avec flatten(applatit array d'objets axes) et boucle pour itérer sur indices des axes

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11), subplot_kw=dict(polar=True))
axes_flat = axes.flatten()  # transforme le tableau 2x2 en tableau 1D de 4 éléments

configs = [
    {"entities": ["Bulbasaur", "Ivysaur", "Venusaur"], "colors": colors_evolution, "title": " Évolution Bulbizarre"},
    {"entities": ["Charmander", "Charmeleon", "Charizard"], "colors": colors_evolution, "title": "Évolution Salamèche"},
    {"entities": ["Squirtle", "Wartortle", "Blastoise"], "colors": colors_evolution, "title": "Évolution Carapuce"},
    {"entities": ["Venusaur", "Charizard", "Blastoise"], "colors": colors_comparatif, "title": "Comparatif formes finales"},
]

for ax, cfg in zip(axes_flat, configs):
    radar_chart(data=df_indexed, axes=ax, cols_to_plot=stats_cols, **cfg)

plt.tight_layout()
plt.show()

### Radar stats par type

In [ ]:
means_by_type = df.groupby(type1)[stats_cols].mean()

In [ ]:
liste_types = means_by_type.index.tolist()

# dico une couleur par type sur une palette qualitative cmap
palette = plt.get_cmap("Paired", len(liste_types))
colors_types = {t: palette(i) for i, t in enumerate(liste_types)}

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 11), subplot_kw=dict(polar=True))
axes_flat = axes.flatten()

configs = [
    {"entities": ["Water", "Fire", "Grass"], "colors": colors_types, "title": ""},
    {"entities": ["Ground", "Flying", "Fighting"], "colors": colors_types, "title": ""},
    {"entities": ["Normal", "Psychic", "Fairy"], "colors": colors_types, "title": ""},
    {"entities": ["Bug", "Dragon", "Poison"], "colors": colors_types, "title": ""},
    {"entities": ["Electric", "Ice", "Dark"], "colors": colors_types, "title": ""},
    {"entities": ["Rock", "Steel", "Ghost"], "colors": colors_types, "title": ""},
]

for ax, cfg in zip(axes_flat, configs):
    radar_chart(data=means_by_type , axes=ax, cols_to_plot=stats_cols, **cfg)

plt.tight_layout()
plt.show()

### Nuage des types sur Attaque / Défense

In [ ]:
types = sorted(type1.unique())
palette = plt.get_cmap("tab20b", len(types))
color_map = {t: palette(i) for i, t in enumerate(types)}
colors = type1.map(color_map)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    df["Attack"],
    df["Defense"],
    s=df["Total"] / 3,   # taille de bulle, divisée pour rester lisible
    c=colors,
    alpha=0.6,
    edgecolors="gray",
    linewidths=1
)

# --- Détection et annotation des outliers ---
# attack_z = (df["Attack"] - df["Attack"].mean()) / df["Attack"].std()
# defense_z = (df["Defense"] - df["Defense"].mean()) / df["Defense"].std()
# is_outlier = (attack_z.abs() > 2) | (defense_z.abs() > 2)
# outliers = df[is_outlier]

outliers = pd.concat([
    df.nlargest(1, "Defense"),
    df.nlargest(1, "Attack"),
    df.nsmallest(1, "Defense"),
    df.nsmallest(1, "Attack"),
])
for _, row in outliers.iterrows():
    ax.annotate(
        row["Name"],
        xy=(row["Attack"], row["Defense"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
        fontweight="bold"
    )
# -----------------------------------------------

ax.set_xlabel("Attack")
ax.set_ylabel("Defense")
ax.set_title("Attack vs Defense (taille des ronds = Total des stats)")

# légende manuelle (scatter ne génère pas de légende de couleur automatiquement, contrairement à seaborn qui le ferait nativement avec hue=)
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map[t], markersize=8, label=t) for t in types]
ax.legend(handles=handles, title="Type", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()